In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

raw_path = Path('../data/raw/chennai_house_price_prediction.csv')

df_raw = pd.read_csv(raw_path)

df_clean = df_raw.copy()

print("Original dataset shape:", df_raw.shape)
print("Cleaning dataset shape:", df_clean.shape)

display(df_clean.head())

Original dataset shape: (7109, 19)
Cleaning dataset shape: (7109, 19)


,PRT_ID,AREA,INT_SQFT,DIST_MAINROAD,N_BEDROOM,N_BATHROOM,N_ROOM,SALE_COND,PARK_FACIL,BUILDTYPE,UTILITY_AVAIL,STREET,MZZONE,QS_ROOMS,QS_BATHROOM,QS_BEDROOM,QS_OVERALL,COMMIS,SALES_PRICE
0,P03210,Karapakkam,1004,131,1.0,1.0,3,AbNormal,Yes,Commercial,AllPub,Paved,A,4.0,3.9,4.9,4.330,144400,7600000
1,P09411,Anna Nagar,1986,26,2.0,1.0,5,AbNormal,No,Commercial,AllPub,Gravel,RH,4.9,4.2,2.5,3.765,304049,21717770
2,P01812,Adyar,909,70,1.0,1.0,3,AbNormal,Yes,Commercial,ELO,Gravel,RL,4.1,3.8,2.2,3.090,92114,13159200
3,P05346,Velachery,1855,14,3.0,2.0,5,Family,No,Others,NoSewr,Paved,I,4.7,3.9,3.6,4.010,77042,9630290
4,P06210,Karapakkam,1226,84,1.0,1.0,3,AbNormal,Yes,Others,AllPub,Gravel,C,3.0,2.5,4.1,3.290,74063,7406250


In [ ]:
# Identify the duplicates
duplicate_rows = df_clean.duplicated().sum()

print("Number of completely duplicated rows:", duplicate_rows)

Number of completely duplicated rows: 0


In [ ]:
df_clean['AREA'].value_counts()

AREA
Chrompet      1681
Karapakkam    1363
KK Nagar       996
Velachery      979
Anna Nagar     783
Adyar          773
T Nagar        496
Chrompt          9
Chrmpet          6
Chormpet         6
TNagar           5
Karapakam        3
Ana Nagar        3
Velchery         2
Ann Nagar        2
Adyr             1
KKNagar          1
Name: count, dtype: int64

In [ ]:
text_columns = df_clean.select_dtypes(include=['object','string']).columns

for col in text_columns:
    df_clean[col] = (df_clean[col].astype('string').str.strip().str.replace(r"\s+", " ", regex=True))


print("Text standardization completed.")

Text standardization completed.


In [ ]:
# Correct Categorical Inconsistencies
category_fixes = {

    "AREA": {
        "Chrompt": "Chrompet",
        "Chrmpet": "Chrompet",
        "Chormpet": "Chrompet",
        "TNagar": "T Nagar",
        "Karapakam": "Karapakkam",
        "Ana Nagar": "Anna Nagar",
        "Ann Nagar": "Anna Nagar",
        "Velchery": "Velachery",
        "Adyr": "Adyar",
        "KKNagar": "KK Nagar"
    },

    "SALE_COND": {
        "Adj Land": "AdjLand",
        "Ab Normal": "AbNormal",
        "Partiall": "Partial",
        "PartiaLl": "Partial"
    },

    "PARK_FACIL": {
        "Noo": "No"
    },

    "BUILDTYPE": {
        "Comercial": "Commercial"
    },

    "UTILITY_AVAIL": {
        "All Pub": "AllPub"
    },

    "STREET": {
        "Pavd": "Paved",
        "NoAccess": "No Access"
    }
}

In [ ]:
# Apply the corrections to the DataFrame
for col, mapping in category_fixes.items():

    df_clean[col] = df_clean[col].replace(mapping)

print("Categorical corrections completed.")

Categorical corrections completed.


In [ ]:
categorical_cols = [
    "AREA",
    "SALE_COND",
    "PARK_FACIL",
    "BUILDTYPE",
    "UTILITY_AVAIL",
    "STREET",
    "MZZONE"
]

for col in categorical_cols:

    print(f"\n")
    print(f"Feature: {col}")


    display(
        df_clean[col]
        .value_counts(dropna=False)
        .rename_axis(col)
        .reset_index(name="Count")
    )



Feature: AREA


,AREA,Count
0,Chrompet,1702
1,Karapakkam,1366
2,KK Nagar,997
3,Velachery,981
4,Anna Nagar,788
5,Adyar,774
6,T Nagar,501




Feature: SALE_COND


,SALE_COND,Count
0,AdjLand,1439
1,Partial,1433
2,Normal Sale,1423
3,AbNormal,1411
4,Family,1403




Feature: PARK_FACIL


...

,UTILITY_AVAIL,Count
0,AllPub,1887
1,NoSeWa,1871
2,NoSewr,1829
3,ELO,1522




Feature: STREET


,STREET,Count
0,Paved,2572
1,Gravel,2520
2,No Access,2017




Feature: MZZONE


,MZZONE,Count
0,RL,1858
1,RH,1822
2,RM,1817
3,C,550
4,A,537
5,I,525


In [ ]:
# Investigating the missing values
missing_report = pd.DataFrame({
    "Missing_Count": df_clean.isnull().sum(),
    "Missing_Percentage": df_clean.isnull().mean() * 100
})

missing_report = missing_report[
    missing_report["Missing_Count"] > 0
].sort_values(
    by="Missing_Count",
    ascending=False
)

display(missing_report)

,Missing_Count,Missing_Percentage
QS_OVERALL,48,0.675200
N_BATHROOM,5,0.070333
N_BEDROOM,1,0.014067


In [ ]:
# Check for invalid numerical values
validation_checks = {

    "INT_SQFT <= 0":
        (df_clean["INT_SQFT"] <= 0).sum(),

    "DIST_MAINROAD < 0":
        (df_clean["DIST_MAINROAD"] < 0).sum(),

    "N_ROOM <= 0":
        (df_clean["N_ROOM"] <= 0).sum(),

    "SALES_PRICE <= 0":
        (df_clean["SALES_PRICE"] <= 0).sum()

}

validation_report = pd.DataFrame.from_dict(
    validation_checks,
    orient="index",
    columns=["Number_of_Rows"]
)

display(validation_report)

,Number_of_Rows
INT_SQFT <= 0,0
DIST_MAINROAD < 0,0
N_ROOM <= 0,0
SALES_PRICE <= 0,0


In [ ]:
output_path = Path('../data/processed/chennai_house_price_prediction_cleaned.csv')

# Create the output directory if it does not exist
output_path.parent.mkdir(parents=True, exist_ok=True)

# Save the cleaned data
df_clean.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully!")
print("Location:", output_path)
print("Final shape:", df_clean.shape)

Cleaned dataset saved successfully!
Location: ..\data\processed\chennai_house_price_prediction_cleaned.csv
Final shape: (7109, 19)
